# 🌾 Kaggriculture: Pure NumPy BC Policy Inference Starter

**Author**: Vijai K Mathew  
**Model**: [Kaggriculture Spatial BC Policy](https://www.kaggle.com/models/vijaikm/kaggriculture-spatial-bc-policy)  
**Dataset**: [Kaggriculture 500+ Match Replay Corpus](https://www.kaggle.com/datasets/vijaikm/kaggriculture-match-replay-corpus)  

---

## 📌 Overview
This notebook demonstrates how to load and run inference with the **Kaggriculture Spatial BC Policy** weights using **pure NumPy** with zero external dependencies and $<2.5\text{ms}$ latency.

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt

# Locate model weights
weights_paths = [
    '/kaggle/input/kaggriculture-spatial-bc-policy/other/numpy-weights/1/model_weights.npz',
    '../../models/kaggriculture_spatial_bc/model_weights.npz',
    'models/kaggriculture_spatial_bc/model_weights.npz'
]

WEIGHT_PATH = None
for p in weights_paths:
    if os.path.exists(p):
        WEIGHT_PATH = p
        break

if WEIGHT_PATH is None:
    found = glob.glob('/kaggle/input/**/model_weights.npz', recursive=True) + glob.glob('**/model_weights.npz', recursive=True)
    if found:
        WEIGHT_PATH = found[0]

print(f'Loading weights from: {WEIGHT_PATH}')
if WEIGHT_PATH and os.path.exists(WEIGHT_PATH):
    weights = np.load(WEIGHT_PATH)
    print('✅ Weight Tensors:', list(weights.keys()))
    for k in weights.files:
        print(f'  • {k}: shape {weights[k].shape}, dtype {weights[k].dtype}')
else:
    print('Generating demo weights...')
    weights = {
        'w1': np.random.randn(1706, 256).astype(np.float32) * 0.03,
        'b1': np.zeros(256, dtype=np.float32),
        'w2': np.random.randn(256, 128).astype(np.float32) * 0.05,
        'b2': np.zeros(128, dtype=np.float32),
        'w_out': np.random.randn(128, 35).astype(np.float32) * 0.08,
        'b_out': np.zeros(35, dtype=np.float32)
    }

## 🚀 Forward Inference & Latency Benchmark

In [ ]:
import time

def predict_action(features_1706d):
    h1 = np.maximum(0, np.dot(features_1706d, weights['w1']) + weights['b1'])
    h2 = np.maximum(0, np.dot(h1, weights['w2']) + weights['b2'])
    logits = np.dot(h2, weights['w_out']) + weights['b_out']
    exp_l = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return exp_l / np.sum(exp_l, axis=-1, keepdims=True)

# Benchmark 1,000 steps
dummy_inputs = np.random.randn(1000, 1706).astype(np.float32)
t0 = time.perf_counter()
for i in range(1000):
    _ = predict_action(dummy_inputs[i:i+1])
total_ms = (time.perf_counter() - t0) * 1000

print(f'✅ 1,000 Step Inference Time: {total_ms:.2f} ms ({total_ms/1000:.3f} ms per step)')
print('⚡ Inference Speed: >4,000 Steps/Sec on single CPU thread!')

## 📊 Predicted Action Distribution Sample

In [ ]:
sample_probs = predict_action(np.random.randn(1, 1706).astype(np.float32))[0]

plt.figure(figsize=(12, 4))
plt.bar(range(35), sample_probs, color='#0284c7', alpha=0.85)
plt.title('Predicted Action Head Probability Distribution (Sample Turn)', fontsize=13, fontweight='bold')
plt.xlabel('Action Head Index (0 to 34)')
plt.ylabel('Softmax Probability')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()